# Kambale graphite deposit: technical appendix

**Author.** Dickson Abdul-Wahab

This appendix documents how the results in `Kambale_Thesis_Results.ipynb` were checked. It is intended for readers who want
to verify or rerun the analysis.


In [ ]:
RUN_CHECKS = True     # runs the automated checks on a copy of the repository
RUN_RERUN = False  # True reruns the full analysis on a copy of the repository (about 15 minutes)


In [ ]:
import html
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display


def find_repo(start=None):
    """Walk up from the working directory until the folder holding pyproject.toml and src/kambale is found."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "kambale").is_dir():
            return candidate
    raise RuntimeError(f"repository root not found above {here}")


REPO = find_repo()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kambale.notebook.env import config_receipts, database_manifest_check, environment_summary
from kambale.notebook.gates import run_gates
from kambale.notebook.receipts import receipt_check
from kambale.notebook.reproduce import reproduce

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 110)
STATUS_NAMES = {"PASS": "passed", "WARN": "flagged", "STOP": "failed"}


def as_frame(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, dict):
        return pd.DataFrame(list(obj.items()), columns=["item", "value"])
    if isinstance(obj, (list, tuple)) and obj and isinstance(obj[0], dict):
        return pd.DataFrame(obj)
    return pd.DataFrame({"value": [str(obj)]})


def callout(text):
    display(HTML('<div style="border:2px solid #b45309;background:#fff7ed;padding:10px 14px;border-radius:6px;'
                 'margin:8px 0">' + text + "</div>"))


## 1. Software environment


In [ ]:
display(as_frame(environment_summary()))


## 2. Analysis settings and input files

Each objective's settings were written to a configuration file before the corresponding analysis was run. Earlier versions
are kept in `config/objective_*/superseded/` with the reason for each change. Each database records a checksum of the
inputs it was built from, so a change to an upstream input is detected.


In [ ]:
display(Markdown("**Configuration files**"))
display(as_frame(config_receipts()))
display(Markdown("**Database checksums**"))
display(as_frame(database_manifest_check()))
display(Markdown("**Input checksums**"))
display(as_frame(receipt_check()))


## 3. Automated checks

Each stage of the analysis runs automated checks on its inputs and outputs. A check either passes, is flagged for attention
(the analysis continues and the issue is reported), or fails (the analysis stops). The checks below are run on a copy of
the repository, so running them does not change any file.


In [ ]:
if RUN_CHECKS:
    checks = as_frame(run_gates()).copy()
    checks["outcome"] = checks["status"].map(lambda s: STATUS_NAMES.get(s, s))
    display(Markdown("**Checks by objective and outcome (number of checks)**"))
    table = pd.crosstab(checks["objective"], checks["outcome"], margins=True)
    display(table)
    display(HTML("<details><summary><b>Full list of checks</b> (click to expand)</summary>"
                 + checks.drop(columns=["status"]).to_html(index=False) + "</details>"))
else:
    display(Markdown("*The automated checks were not run in this execution (`RUN_CHECKS = False`).*"))


### The seven flagged checks

| Objective | Check | What it means |
|---|---|---|
| 1 | Count of `Sgs` intervals | The preliminary thesis design document reported 529 `Sgs` intervals; the database built from the drilling workbook holds 530. The one-interval difference is noted; the database value is used throughout. |
| 2 | Hole orientation | Every hole carries the same planned orientation (dip −60°, azimuth 090°). No downhole surveys exist, so the planned orientation is assumed for all hole geometry. |
| 2 | Low graphitic proportion on one line | Only about 4 per cent of intervals on line N1113000 are graphitic, so continuity estimated for that line rests on few graphitic pairs. |
| 2 | Directions without enough pairs | Lateral directions 000°, 045° and 135° had too few pairs to estimate a variogram. Only the along-line direction could be modelled, so lateral anisotropy is not established. |
| 2 | Stability when one line is left out | Two of ten refits were unstable, both for the fresh-rock variogram. Continuity in fresh rock is therefore the least certain result of Objective 2. |
| 2 | Relative depth only | No collar elevations are available, so all sections are drawn in depth below the collar. |
| 5 | Final model settings | When the selected models were refitted on all data for the 3-D products, tuning chose different settings from the most frequent cross-validation choice (random forest: maximum depth 8 instead of 4, minimum leaf 20 instead of 5; gradient boosting: L2 penalty 0 instead of 1). The validated performance comes from cross-validation; the 3-D products use the refitted models. |


## 4. Full rerun

Setting `RUN_RERUN = True` reruns every stage of Objectives 1 to 5 on a copy of the repository, regenerates all tables and
figures, and compares each regenerated table with the stored one, cell by cell. Timestamps, run times and checksums are
expected to differ and are ignored.


In [ ]:
if RUN_RERUN:
    rep = reproduce(keep=False)
    rep = rep if isinstance(rep, dict) else {"verdict": str(rep)}
    display(Markdown("**Rerun result**"))
    display(Markdown(f"Working copy: `{rep.get('workdir', 'not reported')}`"))
    display(as_frame(rep.get("steps", [])))
    if "tables" in rep:
        display(Markdown("**Table comparison (regenerated against stored)**"))
        display(as_frame(rep["tables"]))
    if "figures" in rep:
        display(Markdown("**Figure comparison**"))
        display(as_frame(rep["figures"]))
    callout("<b>Reproduction verdict:</b> " + html.escape(str(rep.get("verdict", "not reported")))
            + "<br><b>Scope:</b> " + html.escape(str(rep.get("scope_note", ""))))
else:
    display(Markdown(
        "*The rerun was not executed (`RUN_RERUN = False`).* To run it, execute this notebook with `-p RUN_RERUN True`. "
        "It re-runs the pipelines on a copy of the repository (about 15 minutes), compares the regenerated tables and figures "
        "with the stored ones, and never changes the stored databases."))


## 5. Commands

```bash
pip install -e ".[notebook]"          # Jupyter, nbformat, nbclient, papermill and ipykernel
python notebooks/build_notebook.py    # regenerates both notebooks (no outputs) and MATH_NOTES.md
python notebooks/_check_queries.py    # runs every query and code cell of both notebooks (read-only)
papermill notebooks/Kambale_Thesis_Results.ipynb notebooks/output/Kambale_Thesis_Results.ipynb
python -m nbconvert --to html notebooks/output/Kambale_Thesis_Results.ipynb
papermill notebooks/Kambale_Technical_Appendix.ipynb notebooks/output/Kambale_Technical_Appendix.ipynb
python -m nbconvert --to html notebooks/output/Kambale_Technical_Appendix.ipynb
papermill notebooks/Kambale_Technical_Appendix.ipynb notebooks/output/Kambale_Technical_Appendix_rerun.ipynb -p RUN_RERUN True
```

The notebooks read the databases read-only. `notebooks/output/` holds executed copies, which contain query results, and is
not committed. See `notebooks/README.md`.

**Run reports and settings (relative links, from the `notebooks/` folder).**

- Objective 1: [QA/QC report](../reports/QAQC_REPORT.md)
- Objective 2: [run report](../reports/objective_2/RUN_REPORT.md), [README](../reports/objective_2/README.md)
- Objective 3: [run report](../reports/objective_3/RUN_REPORT.md), [README](../reports/objective_3/README.md)
- Objective 4, prediction to a new drill line: [run report](../reports/objective_4/tier_1/RUN_REPORT.md)
- Objective 4, prediction between holes on the same line: [run report](../reports/objective_4/tier_2/RUN_REPORT.md)
- Objective 5: [run report](../reports/objective_5/RUN_REPORT.md)
- Decision logs (superseded settings and the reason for each change): `config/objective_2/superseded/`,
  `config/objective_3/superseded/`, `config/objective_4/superseded/`, `config/objective_5/superseded/`
